In [ ]:
import pandas as pd

# Đọc dữ liệu từ tệp CSV
def clean_data(file_path):
    df = pd.read_csv(file_path)
    
    # Kiểm tra số lượng giá trị bị thiếu ở mỗi cột
    missing_values = df.isnull().sum()
    missing_percentage = (missing_values / len(df)) * 100
    missing_data = pd.DataFrame({'Missing Values': missing_values, 'Percentage': missing_percentage})
    missing_data = missing_data[missing_data['Missing Values'] > 0]
    
    print("Số lượng giá trị bị thiếu ở mỗi cột:")
    print(missing_data.sort_values(by='Missing Values', ascending=False))
    
    
    # Xóa các cột không cần thiết trước
    columns_to_drop = ['Restaurant penalty (Rejection)', 'Review', 'Instructions']
    df = df.drop(columns=columns_to_drop, errors='ignore')
    print("Các cột đã bị xóa:", columns_to_drop)
    
    # Loại bỏ các dòng trùng lặp
    df = df.drop_duplicates()
    
    # Chuyển đổi cột 'Order Placed At' thành kiểu datetime
    df['Order Placed At'] = pd.to_datetime(df['Order Placed At'], format='%I:%M %p, %B %d %Y', errors='coerce')
    
    # Chuyển đổi 'Distance' thành số (loại bỏ km, xử lý "<1km" thành 0.5)
    df['Distance'] = df['Distance'].str.replace('km', '', regex=True)
    df['Distance'] = df['Distance'].replace('<1', 0.5).astype(float)
    
    # Xác định các hàng bị loại bỏ
    threshold = 0.2 * df.shape[1]  # Hơn 20% số cột bị thiếu dữ liệu - 6 cột
    dropped_rows = df[df.isnull().sum(axis=1) > threshold].index.tolist()
    
    # Loại bỏ các hàng có hơn 20% giá trị bị thiếu
    df = df.dropna(thresh=threshold, axis=0)
    
    print("Các hàng bị loại bỏ:", len(dropped_rows))
    
    return df

# Đường dẫn đến tệp dữ liệu
file_path = "order_history_kaggle_data.csv"
cleaned_df = clean_data(file_path)

# Hiển thị thông tin sau khi làm sạch
display(cleaned_df.info())
display(cleaned_df.head())
